In [0]:
%sql
-- Section: Environment Setup
-- Set catalog and schema for SQL queries
USE CATALOG databricks_demo;

USE SCHEMA default;

In [0]:
%sql
-- Section: Filtering Orders with Higher Order Functions
-- Select orders with books having qty >= 3
SELECT 
  y.*
FROM 
  (
    SELECT 
      order_id,
      books,
      FILTER(books, x -> x.qty >= 3) as three_copies
    FROM (
          SELECT 
            order_id,
            from_json(books, 'array<struct<book_id:int,qty:int>>') as books
          FROM tb_orders
        ) x
  ) y 
WHERE SIZE(y.three_copies)> 0 
-- Databricks notebook source

In [0]:
%sql
-- Section: Applying TRANSFORM on Books Array
-- Double book qty for each book in an order
SELECT 
      order_id,
      books,
      TRANSFORM (
        books,
        x -> CAST(x.qty*2 AS INT)
      ) AS quantity
    FROM (
          SELECT 
            order_id,
            from_json(books, 'array<struct<book_id:int,qty:int>>') as books
          FROM tb_orders
        ) x

In [0]:
%sql
-- Section: Creating SQL UDF for Email Domain URL
-- Define get_url function to extract domain and create a URL
CREATE OR REPLACE FUNCTION get_url(email STRING)
RETURNS STRING
RETURN CONCAT('https://www.', split(email, '@')[1])

In [0]:
# Section: Generate Sample Email Data
# Create a Spark DataFrame with sample email records and register as temp view
sample_data = [
    {"id": 1, "domain": "example.com", "path": "/home"},
    {"id": 2, "domain": "testsite.org", "path": "/about"},
    {"id": 3, "domain": "mywebsite.net", "path": "/contact"}
]

df_sample = spark.createDataFrame(sample_data)

# Generate email addresses
from pyspark.sql.functions import concat, lit

df_emails = df_sample.withColumn("email", concat(lit("user@"), df_sample["domain"]))
display(df_emails)

df_emails.createTempView("vm_temp_emails")

In [0]:
%sql
-- Section: Using get_url UDF in SQL
-- Add a URL column for each email address using get_url function
SELECT 
  *,
  get_url(email) as url
FROM vm_temp_emails